In [ ]:
# ============================================================
# ASSIGNMENT 2 - Q2 Option A: GBDT RE-RANKER (MIND, LARGE)
# Team: Shivam Patel + Suyash Pande | CS4.406 IRE
#
# Two-stage retrieve-then-rank. Trains BOTH LightGBM and XGBoost over the Q1 behavioural
# features (from A2_01, loaded from HF). Reports AUC/MRR/nDCG@5/10 BEFORE and AFTER
# re-ranking + extended eval (slices + bootstrap CIs). SAVES both models to /kaggle/working
# AND pushes them to HF.
#
# Loads: mind_large_features.npz (HF donbosoc/mind-artifacts)
# Saves: mind_lgbm_reranker.txt, mind_xgb_reranker.json (output + HF)
# ============================================================
!pip install lightgbm xgboost huggingface_hub -q
import numpy as np
from huggingface_hub import hf_hub_download, HfApi
from kaggle_secrets import UserSecretsClient

HF_REPO = "donbosoc/mind-artifacts"
HF_FILE = "mind_large_features.npz"
tok = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=tok)

path = hf_hub_download(HF_REPO, HF_FILE, repo_type="dataset", token=tok)
d = np.load(path, allow_pickle=True)
Xtr,ytr,gtr = d["Xtr"], d["ytr"], d["gtr"]
Xdv,ydv,gdv = d["Xdv"], d["ydv"], d["gdv"]
FEAT = [str(f) for f in d["feat"]]
gtr = [int(g) for g in gtr]; gdv = [int(g) for g in gdv]
print("loaded:", Xtr.shape, "train |", Xdv.shape, "dev |", len(gtr), "train slates,", len(gdv), "dev slates")


In [ ]:
# ---- metric helpers ----
def _auc(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def _mrr(s,lb):
    for rank,idx in enumerate(np.argsort(-s),1):
        if lb[idx]==1: return 1.0/rank
    return 0.0
def _ndcg(s,lb,k):
    o=np.argsort(-s)[:k];g=lb[o]
    dcg=sum(gg/np.log2(i+2) for i,gg in enumerate(g))
    idcg=sum(gg/np.log2(i+2) for i,gg in enumerate(np.sort(lb)[::-1][:k]))
    return float(dcg/idcg) if idcg>0 else 0.0
def eval_scores(scores,y,g):
    A=[];M=[];N5=[];N10=[];pos=0
    for gi in g:
        s=scores[pos:pos+gi];lb=y[pos:pos+gi];pos+=gi
        a=_auc(s,lb)
        if a is not None: A.append(a)
        M.append(_mrr(s,lb));N5.append(_ndcg(s,lb,5));N10.append(_ndcg(s,lb,10))
    return dict(AUC=np.mean(A),MRR=np.mean(M),nDCG5=np.mean(N5),nDCG10=np.mean(N10))
print("metric helpers ready")


In [ ]:
# ---- BEFORE re-ranking: retrieval order (content similarity proxy) ----
retr_col = FEAT.index("hist_emb_mean")
before = eval_scores(Xdv[:,retr_col], ydv, gdv)
print("BEFORE re-ranking (retrieval order):", {k:round(v,4) for k,v in before.items()})


In [ ]:
# ---- LightGBM re-ranker + SAVE ----
import lightgbm as lgb
lgbm = lgb.LGBMRanker(objective="lambdarank", n_estimators=500, learning_rate=0.03,
                      num_leaves=31, min_child_samples=50, importance_type="gain", verbose=-1)
lgbm.fit(Xtr, ytr, group=gtr)
lgb_scores = lgbm.predict(Xdv)
after_lgb = eval_scores(lgb_scores, ydv, gdv)
print("AFTER (LightGBM):", {k:round(v,4) for k,v in after_lgb.items()})

lgbm.booster_.save_model("/kaggle/working/mind_lgbm_reranker.txt")
api.upload_file(path_or_fileobj="/kaggle/working/mind_lgbm_reranker.txt",
                path_in_repo="mind_lgbm_reranker.txt", repo_id=HF_REPO, repo_type="dataset")
print("saved + pushed mind_lgbm_reranker.txt")

print("feature importance (gain):")
for f,i in sorted(zip(FEAT, lgbm.feature_importances_), key=lambda z:-z[1]):
    print(f"  {f:16s} {i:>10.0f}")


In [ ]:
# ---- XGBoost re-ranker + SAVE ----
import xgboost as xgb
dtr = xgb.DMatrix(Xtr, label=ytr); dtr.set_group(gtr)
ddv = xgb.DMatrix(Xdv, label=ydv); ddv.set_group(gdv)
params = dict(objective="rank:pairwise", eta=0.05, max_depth=6, subsample=0.9,
              colsample_bytree=0.9, eval_metric="ndcg")
xgbm = xgb.train(params, dtr, num_boost_round=400)
xgb_scores = xgbm.predict(ddv)
after_xgb = eval_scores(xgb_scores, ydv, gdv)
print("AFTER (XGBoost):", {k:round(v,4) for k,v in after_xgb.items()})

xgbm.save_model("/kaggle/working/mind_xgb_reranker.json")
api.upload_file(path_or_fileobj="/kaggle/working/mind_xgb_reranker.json",
                path_in_repo="mind_xgb_reranker.json", repo_id=HF_REPO, repo_type="dataset")
print("saved + pushed mind_xgb_reranker.json")


In [ ]:
# ---- BEFORE vs AFTER summary (Q2.4) ----
print("=== BEFORE vs AFTER re-ranking (MIND dev) ===")
print(f"{'metric':8s} {'before':>9} {'LightGBM':>10} {'XGBoost':>9}")
for k in ("AUC","MRR","nDCG5","nDCG10"):
    print(f"{k:8s} {before[k]:>9.4f} {after_lgb[k]:>10.4f} {after_xgb[k]:>9.4f}")


In [ ]:
# ---- EXTENDED EVAL (Q5): slices + bootstrap CIs (LightGBM scores) ----
def per_slate(scores):
    out=[];pos=0
    for gi in gdv:
        out.append((scores[pos:pos+gi], ydv[pos:pos+gi], Xdv[pos:pos+gi])); pos+=gi
    return out
S = per_slate(lgb_scores)
pop_col=FEAT.index("pop_total"); hlen_col=FEAT.index("click_count")
hlens=np.array([s[2][0,hlen_col] for s in S])
P20,P80=np.percentile(hlens,20),np.percentile(hlens,80)

def bci(vals,nb=500):
    v=np.array([x for x in vals if x is not None])
    if len(v)==0: return (0,0,0)
    rng=np.random.default_rng(0)
    m=[rng.choice(v,len(v),replace=True).mean() for _ in range(nb)]
    return v.mean(),np.percentile(m,2.5),np.percentile(m,97.5)

groups={g:{"AUC":[],"MRR":[],"n5":[],"n10":[]} for g in ("all","cold","warm","head","tail")}
for (s,lb,Xs),hl in zip(S,hlens):
    a=_auc(s,lb);mr=_mrr(s,lb);n5=_ndcg(s,lb,5);n10=_ndcg(s,lb,10)
    ci=int(np.argmax(lb)); head = Xs[ci,pop_col] >= np.median(Xs[:,pop_col])
    for b in ["all", ("cold" if hl<=P20 else ("warm" if hl>=P80 else None)), ("head" if head else "tail")]:
        if b is None: continue
        if a is not None: groups[b]["AUC"].append(a)
        groups[b]["MRR"].append(mr);groups[b]["n5"].append(n5);groups[b]["n10"].append(n10)

print("=== Extended eval (LightGBM): AUC 95% CI + slices ===")
for g in ("all","cold","warm","head","tail"):
    m,lo,hi=bci(groups[g]["AUC"])
    print(f"[{g:5s} n={len(groups[g]['MRR'])}] AUC {m:.4f} [{lo:.4f},{hi:.4f}] "
          f"MRR {np.mean(groups[g]['MRR']):.4f} nDCG@5 {np.mean(groups[g]['n5']):.4f} "
          f"nDCG@10 {np.mean(groups[g]['n10']):.4f}")
print("\nModels saved to /kaggle/working AND pushed to HF:", HF_REPO)
